# Credit Risk MLOps — Lesson 3: Preprocessing and Dataset Splitting

This notebook records a group-aware 64%/16%/20% split and leakage-safe preprocessing. Re-run in Kaggle to reproduce the execution.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

dataset = fetch_ucirepo(id=350)
X = dataset.data.features.copy()
y = dataset.data.targets.squeeze().copy()

categorical_features = ["X2", "X3", "X4"]
repayment_features = ["X6", "X7", "X8", "X9", "X10", "X11"]
numerical_features = ["X1", "X5", "X12", "X13", "X14", "X15", "X16", "X17", "X18", "X19", "X20", "X21", "X22", "X23"]

groups = pd.factorize(pd.MultiIndex.from_frame(X))[0]

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_val_idx, test_idx = next(outer_splitter.split(X, y, groups))
X_train_val, y_train_val = X.iloc[train_val_idx], y.iloc[train_val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
groups_train_val = groups[train_val_idx]

inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(inner_splitter.split(X_train_val, y_train_val, groups_train_val))
X_train, y_train = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
X_val, y_val = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

train_global_idx = train_val_idx[train_idx]
val_global_idx = train_val_idx[val_idx]
assert set(groups[train_global_idx]).isdisjoint(set(groups[val_global_idx]))
assert set(groups[train_global_idx]).isdisjoint(set(groups[test_idx]))
assert set(groups[val_global_idx]).isdisjoint(set(groups[test_idx]))

numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numerical", numerical_transformer, numerical_features), ("categorical", categorical_transformer, categorical_features + repayment_features)], remainder="drop")

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
print(X_train_processed.shape, X_val_processed.shape, X_test_processed.shape)
print("Transformed features:", len(preprocessor.get_feature_names_out()))